# CEFR 3-Band Classification - Basic Methods

A survey of **basic, off-the-shelf models**, each run through the same pipeline:

```
features -> model -> probabilities -> 0-100 score (expected value) -> 2 split points -> 3 bands
```

**Models:** Logistic Regression, Naive Bayes, LDA, k-NN, Decision Tree, Random Forest, SVM (RBF),
XGBoost.

**Bands:** `A1-A2` (0) < `B1` (1) < `B2-C1-C2` (2). Baseline to beat: 77%; target >=82%.

For **each** model this notebook reports **train / test / full accuracy**, its **split points**
(the two cut-points on the 0-100 score), and its **score distribution**.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
print("ready | xgboost:", HAS_XGB)

## 1. Load your data  <-- FILL THIS IN

In [ ]:
# TODO: assign your dataframe
df = None
# e.g. df = pd.read_csv("your_file.csv")

## 2. Columns  <-- FILL THIS IN

In [ ]:
FEATURE_COLS = []                 # one feature per group (8-11)
ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL = "ciid", "location", "split", "cefr"
TRAIN_VALUE, TEST_VALUE = "train", "test"
META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42
BAND_MAP = {"A1": 0, "A2": 0, "B1": 1, "B2": 2, "C1": 2, "C2": 2}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3
BAND_ANCHORS = np.array([0.0, 50.0, 100.0])
OPTIMIZE_METRIC = "accuracy"      # or "balanced_accuracy"
BASELINE_ACC, TARGET_ACC = 0.77, 0.82

## 4. Build train / test  (from the `split` column)

In [ ]:
assert df is not None and len(FEATURE_COLS) > 0, "Fill in df and FEATURE_COLS."
miss = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not miss, f"missing columns: {miss}"

def to_band(s):
    s = pd.Series(s)
    if s.dtype.kind in "iuf" and set(pd.unique(s.dropna())) <= {0, 1, 2}:
        return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    m = key.map(BAND_MAP); assert m.notna().all(), f"unmapped: {key[m.isna()].unique()}"
    return m.astype(int).to_numpy()

sp = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_df, test_df = df.loc[sp == TRAIN_VALUE].copy(), df.loc[sp == TEST_VALUE].copy()
X_train, X_test = train_df[FEATURE_COLS].astype(float), test_df[FEATURE_COLS].astype(float)
y_train, y_test = to_band(train_df[LABEL_COL]), to_band(test_df[LABEL_COL])
y_full = np.concatenate([y_train, y_test])

print(f"train/test rows: {len(X_train)}/{len(X_test)} | dropped bad flags: {(~sp.isin([TRAIN_VALUE, TEST_VALUE])).sum()}")
print("train band counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test  band counts:", dict(zip(*np.unique(y_test,  return_counts=True))))

## 5. The shared pipeline + utilities

Each model outputs 3-class probabilities -> 0-100 score (expected value) -> 2 split points
(tuned on train) -> bands. Everything below is applied identically to every model.

In [ ]:
def proba_to_score(proba, anchors=BAND_ANCHORS):
    return np.asarray(proba) @ np.asarray(anchors, float)

def apply_cutpoints(s, t1, t2):
    s = np.asarray(s); return np.where(s <= t1, 0, np.where(s <= t2, 1, 2))

def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score

def fit_cutpoints(s, y, metric=OPTIMIZE_METRIC, ngrid=120):
    s = np.asarray(s); sc = _scorer(metric)
    cand = np.unique(np.percentile(s, np.linspace(0, 100, ngrid)))
    best_v, best = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = sc(y, apply_cutpoints(s, cand[i], cand[j]))
            if v > best_v:
                best_v, best = v, (float(cand[i]), float(cand[j]))
    return best

def make_pipe(estimator):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("model", estimator)])

# ---- Beta reshaping (quantile -> Beta(5,5) bell), fitted on train scores ----
from sklearn.preprocessing import QuantileTransformer
from scipy.stats import beta as _beta
BETA_A = 5.0; _EPS = 1e-3
def fit_beta(s_train):
    a = np.asarray(s_train, float).reshape(-1, 1)
    return QuantileTransformer(output_distribution="uniform", n_quantiles=min(len(a), 1000),
                               subsample=1000000000, random_state=RANDOM_STATE).fit(a)
def to_beta(qt, s):
    p = np.clip(qt.transform(np.asarray(s, float).reshape(-1, 1)).ravel(), _EPS, 1 - _EPS)
    return 100.0 * _beta.ppf(p, BETA_A, BETA_A)

MODELS = {
    "LogisticReg":   LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced"),
    "NaiveBayes":    GaussianNB(),
    "LDA":           LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
    "kNN":           KNeighborsClassifier(n_neighbors=11, weights="distance"),
    "DecisionTree":  DecisionTreeClassifier(max_depth=4, min_samples_leaf=8,
                                            class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest":  RandomForestClassifier(n_estimators=600, min_samples_leaf=3,
                                            class_weight="balanced_subsample",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "SVM-RBF":       SVC(kernel="rbf", C=3.0, gamma="scale", probability=True,
                         class_weight="balanced", random_state=RANDOM_STATE),
}
if HAS_XGB:
    MODELS["XGBoost"] = XGBClassifier(
        objective="multi:softprob", num_class=N_BANDS, eval_metric="mlogloss",
        n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=3,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
        tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)

RESULTS, FITTED = [], {}
print("pipeline ready |", len(MODELS), "models:", list(MODELS))

## 6. Run every model

Fit -> probabilities -> 0-100 score -> split points (train) -> bands, for each model.
Records train / test / full accuracy and the two split points.

In [ ]:
for name, est in MODELS.items():
    pipe = make_pipe(est).fit(X_train, y_train)
    p_tr, p_te = pipe.predict_proba(X_train), pipe.predict_proba(X_test)
    s_tr, s_te = proba_to_score(p_tr), proba_to_score(p_te)
    t1, t2 = fit_cutpoints(s_tr, y_train)
    pred_tr, pred_te = apply_cutpoints(s_tr, t1, t2), apply_cutpoints(s_te, t1, t2)
    pred_full = np.concatenate([pred_tr, pred_te])

    qt = fit_beta(s_tr)                                        # Beta bell fitted on train scores
    bell_full = to_beta(qt, np.concatenate([s_tr, s_te]))
    bell_cuts = tuple(float(x) for x in to_beta(qt, np.array([t1, t2])))

    RESULTS.append(dict(
        model=name,
        train_acc=accuracy_score(y_train, pred_tr),
        test_acc=accuracy_score(y_test, pred_te),
        full_acc=accuracy_score(y_full, pred_full),
        test_bal=balanced_accuracy_score(y_test, pred_te),
        test_qwk=cohen_kappa_score(y_test, pred_te, weights="quadratic"),
        split1=round(t1, 1), split2=round(t2, 1),
    ))
    FITTED[name] = dict(score_full=np.concatenate([s_tr, s_te]), bell_full=bell_full,
                        cuts=(t1, t2), bell_cuts=bell_cuts,
                        pred_full=pred_full, pred_test=pred_te)
    print(f"  {name:<13} train {RESULTS[-1]['train_acc']:.3f} | test {RESULTS[-1]['test_acc']:.3f} "
          f"| full {RESULTS[-1]['full_acc']:.3f} | splits {t1:.1f}/{t2:.1f} -> bell {bell_cuts[0]:.1f}/{bell_cuts[1]:.1f}")

## 7. Leaderboard - accuracy + split points

`train_acc` / `full_acc` are optimistic diagnostics; **`test_acc` is the honest number** to
report against the 77% baseline. `split1` / `split2` are each model's two cut-points on the
0-100 score.

In [ ]:
lb = pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False).reset_index(drop=True)
lb["vs_baseline"] = (lb["test_acc"] - BASELINE_ACC).round(3)
lb["status"] = np.where(lb["test_acc"] >= TARGET_ACC, "PASS >=82%",
                 np.where(lb["test_acc"] >= BASELINE_ACC, "beats 77%", "below 77%"))
display(lb[["model", "train_acc", "test_acc", "full_acc", "test_bal", "test_qwk",
            "split1", "split2", "vs_baseline", "status"]].round(3))
print(f"\nbaseline {BASELINE_ACC:.0%} | target {TARGET_ACC:.0%} | best: "
      f"{lb.iloc[0]['model']} ({lb.iloc[0]['test_acc']:.3f})")

## 8. Score distribution per model - raw vs Beta bell

For **each** model a pair of charts: the raw 0-100 score (left) and the Beta-reshaped bell
(right), with the split points (boundaries) marked as dashed lines.

In [ ]:
try:
    import matplotlib.pyplot as plt
    names = list(FITTED)
    fig, axes = plt.subplots(len(names), 2, figsize=(10, 2.5 * len(names)), squeeze=False)
    for i, name in enumerate(names):
        f = FITTED[name]
        axes[i, 0].hist(f["score_full"], bins=20, range=(0, 100), color="#c0553b")
        for c in f["cuts"]: axes[i, 0].axvline(c, color="k", ls="--", lw=1.3)
        axes[i, 0].set_title(f"{name}: raw  (splits {f['cuts'][0]:.0f}/{f['cuts'][1]:.0f})", fontsize=9)
        axes[i, 1].hist(f["bell_full"], bins=20, range=(0, 100), color="#3b6ea5")
        for c in f["bell_cuts"]: axes[i, 1].axvline(c, color="k", ls="--", lw=1.3)
        axes[i, 1].set_title(f"{name}: bell  (splits {f['bell_cuts'][0]:.0f}/{f['bell_cuts'][1]:.0f})", fontsize=9)
    for a in axes.ravel():
        a.set_xlim(0, 100); a.set_xlabel("0-100 score")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 9. Export: `basic methods.csv`

Per learner: id, location, split, label, and for **every model** its raw 0-100 score, its
Beta-reshaped (bell) score, and its predicted band. Saved to `basic methods.csv`.

In [ ]:
n_tr = len(X_train)
csv = pd.DataFrame({
    ID_COL:     np.concatenate([train_df[ID_COL].values, test_df[ID_COL].values]),
    "location": np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":    ["train"] * n_tr + ["test"] * len(X_test),
    "label":    [BAND_NAMES[i] for i in y_full],
})
for name in FITTED:
    f = FITTED[name]
    csv[f"{name}_raw"]  = np.round(f["score_full"], 2)
    csv[f"{name}_bell"] = np.round(f["bell_full"], 2)
    csv[f"{name}_pred"] = [BAND_NAMES[i] for i in f["pred_full"]]

csv.to_csv("basic methods.csv", index=False)
print("saved 'basic methods.csv' |", len(csv), "rows |", csv.shape[1], "columns")
print("columns:", list(csv.columns))
with pd.option_context("display.max_rows", 400, "display.max_columns", 80):
    display(csv.head(10))

## Notes

- **Same pipeline for all:** every model goes probabilities -> expected-value 0-100 score ->
  tuned split points -> bands, so the comparison is apples-to-apples.
- **Report `test_acc`** (held out). `train_acc`/`full_acc` are optimistic fit diagnostics.
- **Split points differ per model** because each model's score distribution is different.
- Note: LogisticReg is a regression method - included as a basic reference. Trees (DecisionTree,
  RandomForest) are unaffected by the scaler; it is applied uniformly for simplicity.
- The score is the raw expected value (often U-shaped). For a bell, see the reshaping notebook.